In [2]:
import requests
import os

def predecir_estructura_3d(secuencia, nombre_archivo="proteina_predicha"):
    """
    Envía una secuencia de aminoácidos a la API de ESMFold 
    y guarda el resultado en un archivo .pdb
    """
    url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
    
    print(f"Enviando secuencia de {len(secuencia)} residuos...")
    
    # Realizar la petición POST
    response = requests.post(url, data=secuencia)
    
    if response.status_code == 200:
        pdb_content = response.text
        filename = f"{nombre_archivo}.pdb"
        
        with open(filename, "w") as f:
            f.write(pdb_content)
            
        print(f"Estructura guardada exitosamente en: {filename}")
        return filename
    else:
        print(f"Error en la predicción: {response.status_code}")
        return None

In [4]:
def compare_distance_matrices(predicted_matrix_path, target_matrix_path):
    """
    Compara dos matrices de distancias almacenadas en formato CSV 
    y devuelve los valores de error RMSE y MAE.
    """
    # Cargar los archivos CSV
    df_pred = pd.read_csv(predicted_matrix_path)
    df_target = pd.read_csv(target_matrix_path)
    
    # Convertir a numpy arrays para cálculos vectorizados rápidos
    mat_pred = df_pred.to_numpy()
    mat_target = df_target.to_numpy()
    
    # Verificar que las dimensiones coincidan
    if mat_pred.shape != mat_target.shape:
        raise ValueError(f"Error: Las dimensiones no coinciden. Predicha: {mat_pred.shape}, Objetivo: {mat_target.shape}")
    
    # Calcular el Error Cuadrático Medio (MSE) y la Raíz (RMSE)
    mse = np.mean((mat_pred - mat_target) ** 2)
    rmse = np.sqrt(mse)
    
    # Calcular el Error Absoluto Medio (MAE)
    mae = np.mean(np.abs(mat_pred - mat_target))
    
    return {"RMSE": rmse, "MAE": mae}

In [5]:
# 1. Definir secuencias y rutas
mi_secuencia = "RFMYCYWCFRNFKKSQKPNHMRKRSG"
nombre_base = "1aay_prediccion"
ruta_matriz_objetivo = "real_distances/1aay.csv" # Reemplazar con la ruta real

# 2. Obtener estructura 3D (devuelve el nombre del archivo .pdb)
archivo_pdb = predecir_estructura_3d(mi_secuencia, nombre_base)

if archivo_pdb:
    # 3. Extraer coordenadas y generar el archivo .csv
    archivo_csv_predicho = f"{nombre_base}.csv"
    longitud = compute_real_distance_matrix(archivo_pdb, archivo_csv_predicho)
    
    # 4. Evaluar el error comparando ambas matrices
    errores = compare_distance_matrices(archivo_csv_predicho, ruta_matriz_objetivo)
    
    print(f"Secuencia procesada ({longitud} residuos).")
    print(f"Error de la predicción - RMSE: {errores['RMSE']:.4f}, MAE: {errores['MAE']:.4f}")
else:
    print("La predicción de la estructura falló. No se puede calcular el error.")

Enviando secuencia de 26 residuos...
Estructura guardada exitosamente en: 1aay_prediccion.pdb


NameError: name 'compute_real_distance_matrix' is not defined